In [2]:
from google.colab import drive, userdata; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/c2_htr'; RAG = f'{ROOT}/rag'
!pip -q install google-genai rapidfuzz jiwer
import os, sys, json, re, time, joblib, numpy as np, pandas as pd
os.makedirs(f'{ROOT}/code', exist_ok=True); sys.path.append(f'{ROOT}/code')
from google import genai
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print([m.name for m in client.models.list() if 'flash' in m.name])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-pr

In [12]:
%%writefile /content/drive/MyDrive/c2_htr/code/c2_correction.py
"""Component 2 - uncertainty-scoped, retrieval-grounded LLM correction with a verifier gate."""
import re, json, time, unicodedata
from rapidfuzz.distance import Levenshtein

SIN = re.compile(r'[\u0D80-\u0DFF]'); DIG = re.compile(r'\d')
ID_RE = re.compile(r'l\d+w\d+')                # word ids look like l3w5 (line 3, word 5)
PUNCT = '"\'.,;:!?()[]{}<>|/\\-–—_*#~`“”‘’'
norm = lambda t: unicodedata.normalize('NFC', str(t))
tokens = lambda t: [w.strip(PUNCT) for w in norm(t).split() if w.strip(PUNCT)]
# consonant skeleton: drop vowel signs, virama (්), ං ඃ and the joiner -> the consonants that identify the word.
# Handwriting OCR mostly confuses the small vowel strokes, so R2 compares skeletons.
VOWEL_MARKS = re.compile(r'[\u0DCA-\u0DDF\u0DF2\u0DF3\u0D82\u0D83\u200d]')
skeleton = lambda s: VOWEL_MARKS.sub('', s)

# ---------- 1. uncertainty marking from CRNN character confidences ----------
def words_with_conf(text, confs):
    out, i = [], 0
    for w in text.split(' '):
        c = confs[i:i + len(w)]; i += len(w) + 1
        if w: out.append((w, min(c) if c else 0.0))
    return out

def build_page(lines, tau):
    """lines = [(ocr_text, char_confidences), ...] -> [[{id,text,conf,unsure}, ...], ...]"""
    return [[dict(id=f'l{li}w{wi}', text=w, conf=c, unsure=c < tau)
             for wi, (w, c) in enumerate(words_with_conf(norm(t), conf))]
            for li, (t, conf) in enumerate(lines)]

def render(page, edits=None, marked=False):
    edits = edits or {}
    return '\n'.join(' '.join(
        f"«{w['id']}:{w['text']}»" if marked and w['unsure'] and w['id'] not in edits else edits.get(w['id'], w['text'])
        for w in line) for line in page)

# ---------- 2. prompts ----------
PROMPT_SCOPED = """You fix OCR errors in ONE handwritten civic complaint from Sri Lanka (Sinhala, English or mixed).
Words the OCR was unsure about look like «ID:word» (for example «l2w5:පදුලි» has ID l2w5). Everything else is reliable.
Line breaks only show layout.

RULES
1. Only propose replacements for «ID:word» items. Never touch anything else.
2. Propose a fix only if supported by: a REFERENCE complaint, the LEXICON, the same word elsewhere in this
   complaint, the DESCRIPTION, or the obvious spelling of a common Sinhala/English word.
3. The replacement must look like the OCR word (a few wrong or missing letters). Do not translate,
   summarise or add information. If unsure, leave it out.
4. Never change digits, dates, numbers or codes unless the exact value appears in the evidence.
5. Use the ID exactly as written (e.g. "l2w5"), without « » or the word.
6. Return JSON only: {{"edits": [{{"id": "l2w5", "to": "...", "evidence": "reference|lexicon|complaint|description|spelling"}}]}}

DESCRIPTION (typed by the citizen): {description}

REFERENCE (similar past complaints):
{refs}

LEXICON: {lexicon}

COMPLAINT (OCR):
{text}
{feedback}"""

PROMPT_BLIND = """Correct the OCR errors in this handwritten Sri Lankan civic complaint (Sinhala/English).
Keep the same number of lines. Return JSON only: {{"lines": ["...", "..."]}}
{refs}
OCR TEXT:
{text}"""

def call_llm(client, model, prompt, retries=4):
    for a in range(retries):
        try:
            r = client.models.generate_content(model=model, contents=prompt,
                    config={'temperature': 0, 'response_mime_type': 'application/json'})
            return json.loads(re.search(r'\{.*\}', r.text or '', re.S).group(0))
        except Exception as e:
            if a == retries - 1: raise
            time.sleep(10 * (a + 1))          # rate limit / temporary error -> wait and retry

# ---------- 3. verifier (rules R1-R4) ----------
def check(w, to, vocab, use_verifier=True, max_ratio=0.5):
    if w is None: return ['R1_unknown_id']
    r = []
    if not w['unsure']: r.append('R1_not_uncertain')
    if not to or len(to.split()) > 2: r.append('R1_bad_replacement')
    if use_verifier and not r:
        a, b = skeleton(w['text']), skeleton(to)                                 # R2 on consonant skeletons
        if Levenshtein.distance(a, b) > max(1, max_ratio * max(len(a), len(b))):
            r.append('R2_too_different')
        tk = tokens(to)
        if not tk or any(t not in vocab for t in tk): r.append('R3_unsupported')
        if DIG.findall(w['text']) != DIG.findall(to) and to not in vocab: r.append('R4_digits_changed')
    return r

# ---------- 4. the full loop: mark -> LLM edit list -> verify -> re-check ----------
def correct_page(lines, client, model, refs, lexicon, tau=0.8, use_verifier=True, max_rounds=3,
                 description='', sleep=4):
    page = build_page(lines, tau)
    by_id = {w['id']: w for line in page for w in line}
    raw = render(page)
    base = dict(raw=raw, refs=refs, edits=[], prompts=[], rounds=0)
    if not any(w['unsure'] for w in by_id.values()):
        return {**base, 'final': raw, 'decision': 'NO_CHANGES'}
    vocab = (set(lexicon) | {w['text'] for w in by_id.values() if not w['unsure']}
             | {t for d in refs for t in tokens(d)} | set(tokens(description)))
    lex_txt = ', '.join(lexicon) if lexicon else '(none)'
    refs_txt = '\n'.join(f'- {d}' for d in refs) or '(none)'
    accepted, log, prompts, feedback, rnd = {}, [], [], '', 0
    for rnd in range(1, max_rounds + 1):
        prompt = PROMPT_SCOPED.format(description=description or '(none)', refs=refs_txt, lexicon=lex_txt,
                                      text=render(page, accepted, marked=True),
                                      feedback=f'\nFEEDBACK ON YOUR PREVIOUS EDITS (fix or drop):\n{feedback}' if feedback else '')
        prompts.append(prompt)
        try:
            out = call_llm(client, model, prompt); time.sleep(sleep)
        except Exception as e:
            return {**base, 'final': raw, 'edits': log, 'prompts': prompts, 'rounds': rnd,
                    'decision': 'FALLBACK_RAW', 'error': str(e)}
        rej, n = [], 0
        for e in out.get('edits', []):
            m = ID_RE.search(str(e.get('id', ''))); eid = m.group(0) if m else str(e.get('id', ''))
            to = norm(e.get('to', '')).strip(); w = by_id.get(eid)
            if eid in accepted or (w and to == w['text']): continue
            reasons = check(w, to, vocab, use_verifier); n += 1
            log.append(dict(id=eid, frm=w['text'] if w else '', to=to, evidence=e.get('evidence', ''),
                            ok=not reasons, reasons=reasons, round=rnd))
            if reasons: rej.append(log[-1])
            else: accepted[eid] = to
        if not use_verifier or not rej or len(rej) / max(1, n) <= 0.3: break
        feedback = '\n'.join(f"- {r['id']}: '{r['frm']}' -> '{r['to']}' rejected ({', '.join(r['reasons'])})" for r in rej)
    nrej = sum(not l['ok'] for l in log)
    decision = ('ACCEPT' if accepted and not nrej else 'PARTIAL' if accepted else 'REVIEW' if nrej else 'NO_CHANGES')
    return {**base, 'final': render(page, accepted), 'edits': log, 'prompts': prompts, 'rounds': rnd, 'decision': decision}

# ---------- 5. ablation baseline: LLM rewrites everything (no marks, no verifier) ----------
def correct_blind(lines, client, model, refs=()):
    raw = '\n'.join(norm(t) for t, _ in lines)
    refs_txt = ('REFERENCE (similar past complaints):\n' + '\n'.join(f'- {d}' for d in refs) + '\n') if refs else ''
    try:
        out = call_llm(client, model, PROMPT_BLIND.format(refs=refs_txt, text=raw))
        return dict(raw=raw, final='\n'.join(norm(x) for x in out.get('lines', [])), decision='BLIND')
    except Exception as e:
        return dict(raw=raw, final=raw, decision='FALLBACK_RAW', error=str(e))

Overwriting /content/drive/MyDrive/c2_htr/code/c2_correction.py


In [17]:
import importlib, c2_correction as C; importlib.reload(C)
from rapidfuzz import fuzz
MODEL = 'models/gemini-3.5-flash'          # <- from the Step 1 list

pr = pd.read_csv(f'{ROOT}/test_predictions.csv', keep_default_na=False)
pr['conf'] = pr.conf.map(json.loads)
pr['page'] = pr.file.str.replace(r'_c\d+_l\d+\.png$', '', regex=True)
pr = pr.sort_values('file')                               # reading order: column, then line

docs = [json.loads(l) for l in open(f'{RAG}/past_complaints.jsonl', encoding='utf-8')]
texts = [d['text'] for d in docs]
tfv, TF = joblib.load(f'{RAG}/tfidf.joblib')
lexicon = open(f'{RAG}/lexicon_si.txt', encoding='utf-8').read().split('\n')
from sklearn.metrics.pairwise import linear_kernel

def retrieve(query, truth_lines=(), k=20):
    leak = {i for i, t in enumerate(texts) if C.SIN.search(t) and any(fuzz.ratio(t, tl) >= 90 for tl in truth_lines)}
    s = linear_kernel(tfv.transform([query]), TF).ravel()
    return [texts[i] for i in s.argsort()[::-1][:k + len(leak)] if i not in leak][:k]

pages = {p: g for p, g in pr.groupby('page')}
print({s: pr[pr.split == s].page.nunique() for s in ['val', 'test']}, 'pages | lexicon', len(lexicon))

{'val': 4, 'test': 3} pages | lexicon 583


In [10]:
val = pr[pr.split == 'val']
rows = []
for tau in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95):
    tp = fp = fn = 0
    for _, r in val.iterrows():
        truth = set(C.tokens(r.truth))
        for w, c in C.words_with_conf(C.norm(r.pred), r.conf):
            wrong = w.strip(C.PUNCT) not in truth
            if c < tau: tp += wrong; fp += (not wrong)
            else: fn += wrong
    rows.append((tau, tp / max(1, tp + fn), tp / max(1, tp + fp), (tp + fp)))
print(pd.DataFrame(rows, columns=['tau', 'wrong words caught (recall)', 'marked words that are wrong (precision)', 'words marked']).round(3).to_string(index=False))

 tau  wrong words caught (recall)  marked words that are wrong (precision)  words marked
0.50                        0.558                                    0.919            99
0.60                        0.736                                    0.909           132
0.70                        0.822                                    0.887           151
0.80                        0.896                                    0.874           167
0.90                        0.951                                    0.833           186
0.95                        0.969                                    0.814           194


In [18]:
import jiwer
TAU = 0.8                                   # <- from Step 4
results = []
for p, g in pages.items():
    if g.split.iloc[0] != 'val': continue
    lines = list(zip(g.pred, g.conf))
    refs = retrieve(' '.join(g.pred), list(g.truth))
    out = C.correct_page(lines, client, MODEL, refs, lexicon, tau=TAU)
    truth = ' '.join(g.truth); raw = ' '.join(out['raw'].split('\n')); fin = ' '.join(out['final'].split('\n'))
    results.append(dict(page=p, cer_raw=jiwer.cer(truth, raw), cer_final=jiwer.cer(truth, fin), **out))
    print(f"\n=== {p} | {out['decision']} | CER {results[-1]['cer_raw']:.3f} -> {results[-1]['cer_final']:.3f}")
    for e in out['edits']: print(('  KEPT    ' if e['ok'] else '  REVERTED'), e['frm'], '->', e['to'], e['reasons'] or '', f"[{e['evidence']}]")

json.dump(results, open(f'{ROOT}/llm_val_runs.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)  # evidence: prompts + answers
print('\nmean CER raw', np.mean([r['cer_raw'] for r in results]).round(3), '-> final', np.mean([r['cer_final'] for r in results]).round(3))


=== CamScanner_10-02-2026_10_07_p013 | PARTIAL | CER 0.225 -> 0.200
  KEPT     දරකංන -> දරන  [lexicon]
  KEPT     අකය -> අංකය  [lexicon]
  KEPT     හෝ්කනිදර| -> හෝකන්දර  [lexicon]
  KEPT     ඇත් -> ගත්  [reference]
  KEPT     ක්‍රීයා -> ක්‍රියා  [lexicon]
  KEPT     පඉටු -> ඉටු  [reference]
  REVERTED ප -> ඇත ['R2_too_different'] [reference]

=== CamScanner_10-02-2026_10_07_p018 | ACCEPT | CER 0.290 -> 0.277
  KEPT     පදුලි -> විදුලි  [lexicon]
  KEPT     බබුරු -> බුබුලු  [lexicon]
  KEPT     බත්තරටුල්ල -> බත්තරමුල්ල  [lexicon]
  KEPT     අන්‍රීය -> අක්‍රීය  [lexicon]
  KEPT     මාවක -> මාවත  [lexicon]

=== CamScanner_10-02-2026_10_31_p003 | PARTIAL | CER 0.607 -> 0.568
  KEPT     සර්ගය -> මාර්ගය  [lexicon]
  KEPT     පළිසසර -> පිළිසකර  [lexicon]
  KEPT     පිංාර්ධනය -> සංවර්ධනය  [lexicon]
  KEPT     ගැනී -> ගැනීම  [lexicon]
  REVERTED සගා -> සදහා ['R2_too_different'] [lexicon]
  KEPT     කප්පසක් -> තාප්පයක්  [lexicon]
  KEPT     සම්බජ්යව -> සම්බන්ධව  [lexicon]
  KEPT     ළ්න්රය -> ද